# Google Merchandise Store Funnel and Conversion Analysis

## SQL Analysis

This notebook uses SQL to inspect the three source datasets, clean the event table, and answer business questions about funnel performance, countries, brands, recorded customer value and purchase-session revenue.

### Import libraries

In [1]:
import pandas as pd
import sqlite3

In [2]:

events_raw = pd.read_csv( "events1.csv")

items = pd.read_csv("items.csv")

users = pd.read_csv("users.csv")

print("Raw events:", events_raw.shape)
print("Items:", items.shape)
print("Users:", users.shape)

Raw events: (758884, 7)
Items: (1381, 6)
Users: (270154, 3)


### Create a temporary database

In [3]:
conn = sqlite3.connect(":memory:")

events_raw.to_sql(
    "events_raw",
    conn,
    if_exists="replace",
    index=False
)

items.to_sql(
    "items",
    conn,
    if_exists="replace",
    index=False
)

users.to_sql(
    "users",
    conn,
    if_exists="replace",
    index=False
)

270154

### Create a Helper function

In [4]:
def run_query(query):
    return pd.read_sql_query(query, conn)

## 1. Data Structure and Initial Checks

Before cleaning the event data, I checked the size of each table, missing values in the event fields and exact duplicate rows.

In [5]:
table_size_query = """
SELECT
    'events_raw' AS table_name,
    COUNT(*) AS rows
FROM events_raw

UNION ALL

SELECT
    'items' AS table_name,
    COUNT(*) AS rows
FROM items

UNION ALL

SELECT
    'users' AS table_name,
    COUNT(*) AS rows
FROM users;
"""

run_query(table_size_query)

,table_name,rows
0,events_raw,758884
1,items,1381
2,users,270154


### Check Missing values

In [6]:
missing_values_query = """
SELECT
    SUM(CASE WHEN user_id IS NULL THEN 1 ELSE 0 END)
        AS missing_user_id,
    SUM(CASE WHEN ga_session_id IS NULL THEN 1 ELSE 0 END)
        AS missing_session_id,
    SUM(CASE WHEN country IS NULL THEN 1 ELSE 0 END)
        AS missing_country,
    SUM(CASE WHEN device IS NULL THEN 1 ELSE 0 END)
        AS missing_device,
    SUM(CASE WHEN type IS NULL THEN 1 ELSE 0 END)
        AS missing_event_type,
    SUM(CASE WHEN item_id IS NULL THEN 1 ELSE 0 END)
        AS missing_item_id,
    SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END)
        AS missing_date
FROM events_raw;
"""

run_query(missing_values_query)

,missing_user_id,missing_session_id,missing_country,missing_device,missing_event_type,missing_item_id,missing_date
0,0,0,4555,0,0,0,0


### Check exact duplicate event rows

In [7]:
duplicate_check_query = """
WITH duplicate_groups AS (
    SELECT
        user_id,
        ga_session_id,
        country,
        device,
        type,
        item_id,
        date,
        COUNT(*) AS occurrence_count
    FROM events_raw
    GROUP BY
        user_id,
        ga_session_id,
        country,
        device,
        type,
        item_id,
        date
    HAVING COUNT(*) > 1
)

SELECT
    COUNT(*) AS duplicate_groups,
    SUM(occurrence_count - 1)
        AS duplicate_rows_to_remove
FROM duplicate_groups;
"""

run_query(duplicate_check_query)

,duplicate_groups,duplicate_rows_to_remove
0,28414,39498


## 2. Data Cleaning

The event data contained exact duplicate rows and missing country values. Exact duplicates were removed, while missing countries were labelled as `Unknown` so the remaining event information could still be used.


In [8]:
clean_events_sql = """
DROP TABLE IF EXISTS events_clean;

CREATE TABLE events_clean AS

SELECT DISTINCT
    user_id,
    ga_session_id,
    COALESCE(country, 'Unknown') AS country,
    device,
    type,
    item_id,
    date
FROM events_raw;
"""

conn.executescript(clean_events_sql)

print("Clean event table created.")

Clean event table created.


In [9]:
cleaning_validation_query = """
SELECT
    COUNT(*) AS cleaned_rows,

    SUM(CASE WHEN country IS NULL THEN 1 ELSE 0 END)
        AS missing_country,

    COUNT(*) - (
        SELECT COUNT(*)
        FROM (
            SELECT DISTINCT *
            FROM events_clean
        )
    ) AS exact_duplicate_rows

FROM events_clean;
"""

run_query(cleaning_validation_query)

,cleaned_rows,missing_country,exact_duplicate_rows
0,719386,0,0


## 3. Funnel Performance

How does event volume change from add to cart to checkout and purchase?

In [10]:
funnel_summary_query = """
WITH funnel_counts AS (
    SELECT
        SUM(
            CASE WHEN type = 'add_to_cart'
                 THEN 1 ELSE 0 END
        ) AS add_to_cart_events,

        SUM(
            CASE WHEN type = 'begin_checkout'
                 THEN 1 ELSE 0 END
        ) AS begin_checkout_events,

        SUM(
            CASE WHEN type = 'purchase'
                 THEN 1 ELSE 0 END
        ) AS purchase_events

    FROM events_clean
)

SELECT
    add_to_cart_events,
    begin_checkout_events,
    purchase_events,

    ROUND(
        begin_checkout_events * 100.0
        / NULLIF(add_to_cart_events, 0),
        2
    ) AS cart_to_checkout_rate_pct,

    ROUND(
        purchase_events * 100.0
        / NULLIF(begin_checkout_events, 0),
        2
    ) AS checkout_to_purchase_rate_pct,

    ROUND(
        purchase_events * 100.0
        / NULLIF(add_to_cart_events, 0),
        2
    ) AS purchase_to_cart_rate_pct

FROM funnel_counts;
"""

run_query(funnel_summary_query)

,add_to_cart_events,begin_checkout_events,purchase_events,cart_to_checkout_rate_pct,checkout_to_purchase_rate_pct,purchase_to_cart_rate_pct
0,666071,38604,14711,5.8,38.11,2.21


### Funnel Summary

Begin-checkout event volume was only 5.80% of add-to-cart volume, making the gap before checkout the largest funnel-stage drop. Purchase-event volume was 38.11% of checkout-event volume.

*These ratios compare event counts and do not track individual users through the funnel.*

## 4. Country Funnel Summary

Among the countries with the highest add-to-cart activity, which recorded the strongest event-based funnel rates?

In [11]:
country_funnel_query = """
WITH country_funnel AS (
    SELECT
        country,

        SUM(
            CASE WHEN type = 'add_to_cart'
                 THEN 1 ELSE 0 END
        ) AS add_to_cart_events,

        SUM(
            CASE WHEN type = 'begin_checkout'
                 THEN 1 ELSE 0 END
        ) AS begin_checkout_events,

        SUM(
            CASE WHEN type = 'purchase'
                 THEN 1 ELSE 0 END
        ) AS purchase_events

    FROM events_clean
    GROUP BY country
),

top_traffic_countries AS (
    SELECT *
    FROM country_funnel
    ORDER BY add_to_cart_events DESC
    LIMIT 10
)

SELECT
    country,
    add_to_cart_events,
    begin_checkout_events,
    purchase_events,

    ROUND(
        begin_checkout_events * 100.0
        / NULLIF(add_to_cart_events, 0),
        2
    ) AS cart_to_checkout_rate_pct,

    ROUND(
        purchase_events * 100.0
        / NULLIF(add_to_cart_events, 0),
        2
    ) AS purchase_to_cart_rate_pct

FROM top_traffic_countries
ORDER BY purchase_to_cart_rate_pct DESC;
"""

run_query(country_funnel_query)

,country,add_to_cart_events,begin_checkout_events,purchase_events,cart_to_checkout_rate_pct,purchase_to_cart_rate_pct
0,CA,52443,3519,1308,6.71,2.49
1,ES,14085,865,347,6.14,2.46
2,FR,12567,835,309,6.64,2.46
3,DE,10360,521,238,5.03,2.30
4,IN,60072,3614,1343,6.02,2.24
5,GB,20129,1205,442,5.99,2.20
6,US,297416,16626,6506,5.59,2.19
7,CN,12151,591,248,4.86,2.04
8,TW,10871,618,215,5.68,1.98
9,IT,10504,482,157,4.59,1.49


Country Funnel Summary

Among the ten countries with the most add-to-cart activity, Canada recorded the strongest funnel performance, with a 6.71% cart-to-checkout rate and a 2.49% purchase-to-cart rate. Spain and France followed closely at 2.46%.

The United States generated the highest volume by a wide margin, including 6,506 purchase events, but its purchase-to-cart rate was 2.19%. This makes the US the volume leader, while Canada had the strongest conversion rate among these countries.

Italy recorded the lowest purchase-to-cart rate at 1.49%, suggesting weaker funnel progression.

These rates compare event counts and do not track individual users through the funnel.

## 5. Brand Performance Summary

Which brands generate the most purchase activity and recorded product revenue?

In [12]:
brand_performance_query = """
WITH brand_performance AS (
    SELECT
        COALESCE(NULLIF(TRIM(i.brand), ''), 'Unknown') AS brand,
        COUNT(*) AS purchase_events,
        SUM(i.price_in_usd) AS revenue_usd
    FROM events_clean AS e
    LEFT JOIN items AS i
        ON e.item_id = i.id
    WHERE e.type = 'purchase'
    GROUP BY COALESCE(NULLIF(TRIM(i.brand), ''), 'Unknown')
),
totals AS (
    SELECT
        SUM(purchase_events) AS total_purchase_events,
        SUM(revenue_usd) AS total_revenue
    FROM brand_performance
)
SELECT
    bp.brand,
    bp.purchase_events,
    ROUND(bp.purchase_events * 100.0 /
          NULLIF(t.total_purchase_events, 0), 2) AS purchase_event_share_pct,
    ROUND(bp.revenue_usd, 2) AS revenue_usd,
    ROUND(bp.revenue_usd * 100.0 /
          NULLIF(t.total_revenue, 0), 2) AS revenue_share_pct
FROM brand_performance AS bp
CROSS JOIN totals AS t
ORDER BY bp.revenue_usd DESC
LIMIT 10;
"""

run_query(brand_performance_query)

,brand,purchase_events,purchase_event_share_pct,revenue_usd,revenue_share_pct
0,Google,12714,86.43,258230.0,89.23
1,Android,1079,7.33,15706.0,5.43
2,YouTube,566,3.85,8273.0,2.86
3,Google Cloud,140,0.95,4794.0,1.66
4,#IamRemarkable,212,1.44,2404.0,0.83


Brand Performance Summary

Google products clearly dominate the store, accounting for 86.43% of purchase events and 89.23% of recorded product revenue. Android is a distant second, contributing 5.43% of revenue, followed by YouTube at 2.86%.

Although #IamRemarkable recorded more purchase events than Google Cloud, it generated less revenue. This suggests that its purchased products were generally lower-priced.

## 6. Recorded Customer Value and Purchase Behaviour

How does purchase activity differ between users with zero recorded LTV and those with positive recorded LTV?

In [13]:
customer_value_query = """
WITH user_purchase_activity AS (
    SELECT
        u.id AS user_id,
        u.ltv,
        CASE
            WHEN u.ltv > 0 THEN 'Positive LTV'
            ELSE 'Zero LTV'
        END AS ltv_status,
        SUM(
            CASE WHEN e.type = 'purchase' THEN 1 ELSE 0 END
        ) AS purchase_events
    FROM users AS u
    LEFT JOIN events_clean AS e
        ON u.id = e.user_id
    GROUP BY u.id, u.ltv
)
SELECT
    ltv_status,
    COUNT(*) AS total_users,
    SUM(
        CASE WHEN purchase_events > 0 THEN 1 ELSE 0 END
    ) AS users_with_purchase,
    ROUND(
        SUM(CASE WHEN purchase_events > 0 THEN 1 ELSE 0 END)
        * 100.0 / COUNT(*), 2
    ) AS users_with_purchase_pct,
    SUM(purchase_events) AS purchase_events,
  COALESCE(
    ROUND(
        SUM(purchase_events) * 1.0 /
        NULLIF(
            SUM(CASE WHEN purchase_events > 0 THEN 1 ELSE 0 END),
            0
        ), 2
    ), 0
    )AS avg_purchase_events_per_purchasing_user,
    ROUND(SUM(ltv), 2) AS total_recorded_ltv,
    ROUND(AVG(ltv), 2) AS average_recorded_ltv
FROM user_purchase_activity
GROUP BY ltv_status
ORDER BY average_recorded_ltv DESC;
"""

run_query(customer_value_query)

,ltv_status,total_users,users_with_purchase,users_with_purchase_pct,purchase_events,avg_purchase_events_per_purchasing_user,total_recorded_ltv,average_recorded_ltv
0,Positive LTV,4445,4066,91.47,14711,3.62,407626.0,91.7
1,Zero LTV,265709,0,0.00,0,0.00,0.0,0.0


### Customer Value Summary

The LTV field comes directly from `users.csv`. Of the 4,445 users with positive LTV, 4,066 appeared in the purchase events and accounted for all 14,711 purchase events. The remaining 379 may reflect a difference in coverage between the users and events data. Users with zero LTV recorded no purchases.


## 7. Revenue per Purchase Session

What is the average recorded product revenue per purchase session?


Because the dataset has no order ID, we will use each user_id and ga_session_id combination as a purchase session. This is a proxy for an order, not a confirmed order count.

In [14]:
purchase_session_query = """
WITH purchase_sessions AS (
    SELECT
        e.user_id,
        e.ga_session_id,
        COUNT(*) AS purchase_events,
        SUM(COALESCE(i.price_in_usd, 0)) AS session_revenue_usd
    FROM events_clean AS e
    LEFT JOIN items AS i
        ON e.item_id = i.id
    WHERE e.type = 'purchase'
    GROUP BY
        e.user_id,
        e.ga_session_id
)
SELECT
    COUNT(*) AS purchase_sessions,
    SUM(purchase_events) AS purchase_events,
    ROUND(SUM(session_revenue_usd), 2) AS total_recorded_revenue_usd,
    ROUND(AVG(session_revenue_usd), 2) AS average_revenue_per_session_usd,
    ROUND(MIN(session_revenue_usd), 2) AS minimum_session_revenue_usd,
    ROUND(MAX(session_revenue_usd), 2) AS maximum_session_revenue_usd
FROM purchase_sessions;
"""

run_query(purchase_session_query)

,purchase_sessions,purchase_events,total_recorded_revenue_usd,average_revenue_per_session_usd,minimum_session_revenue_usd,maximum_session_revenue_usd
0,4446,14711,289407.0,65.09,1.0,1026.0


### Purchase Session Revenue Summary

The 14,711 purchase events were grouped into 4,446 purchase sessions, generating 289,407 in recorded product revenue. Average revenue per purchase session was 65.09.

*Since the dataset does not contain an order ID, this represents revenue per purchase session rather than confirmed average order value.*

## 8. Final SQL Findings and Limitations

#### Key Findings

- Begin-checkout event volume was only 5.80% of add-to-cart volume, showing that the largest funnel gap occurs before checkout.
- The United States generated the most purchase events, while Canada had the highest purchase-to-cart rate among the ten highest-traffic countries.
- Google products contributed 89.23% of recorded product revenue.
- All purchase events were linked to users with positive LTV in users.csv.
- The store generated 289,407 dollars across 4,446 purchase sessions, averaging 65.09 dollars per session.

#### Limitations
- Funnel rates compare event totals and do not track individual users through each stage.
- The dataset does not contain an order ID, so purchase sessions were used as a proxy.
- Revenue is based on listed product prices and does not include quantities, discounts, taxes, shipping or refunds.
- LTV comes directly from users.csv; it was not calculated or predicted in this analysis.
- The data shows where funnel volume declines but cannot explain why customers leave.